# SOXX 披露成分变化与 Stage1 弱窗口支持度

冻结 Stage1 原生 0bp、Tiered 账户只读诊断。使用原始 N-PORT ISIN 与 SEC 披露后首个 Qlib 交易日；不产生交易规则。

In [1]:
import glob, hashlib, json, os, pathlib, re, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
root = pathlib.Path.cwd()
card = root/'research/stage1_soxx_basket_change_window_support_20261023'
plan = json.loads((card/'plan.json').read_text())
source = {k: json.loads((root/v).read_text()) for k,v in plan['sources'].items() if k!='qlib_provider'}
assert source['nport_identity']['NPORT_identity_gate_passed'] and source['nport_identity']['source_xml_files']==28
assert source['modern_event_recorder']['report_events']==15 and source['training_event_recorder']['report_events']==13
assert source['stage1_component_recorder']['recorder_id']=='2f0392615c4c4b15928b4ada6e01ecd9'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/plan['sources']['qlib_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
cal=pd.DatetimeIndex(D.calendar(start_time='2018-01-01',end_time='2026-10-10',freq='day'))
series=[]
for label in ('training_event_recorder','modern_event_recorder'):
 s=source[label]
 with R.uri_context(s['recorder_uri']):
  rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
  assert rec.status=='FINISHED'
  series.append(rec.load_object('analysis/source_events'))
events=pd.concat(series,ignore_index=True)
assert len(events)==28 and events.accession.is_unique
with R.uri_context(source['stage1_component_recorder']['recorder_uri']):
 s=source['stage1_component_recorder'];rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
 assert rec.status=='FINISHED'
 windows=rec.load_object('analysis/rolling126');daily=rec.load_object('analysis/daily')
assert len(windows)==819 and len(daily)==944
assert pd.DatetimeIndex(daily.signal_date).isin(cal).all()
assert all(windows.iloc[i].first_trade==str(pd.Timestamp(daily.iloc[i].trade_date).date()) and windows.iloc[i].last_exit==str(pd.Timestamp(daily.iloc[i+125].exit_date).date()) for i in range(len(windows)))
patterns=['data/snapshots/stage1_soxx_2019_nport_source_specimen_20261023/*.xml','data/snapshots/stage1_soxx_2019_2022_nport_original_xml_20261023/*.xml','data/snapshots/stage1_soxx_2023_2026_nport_original_source_20261023/*.xml']
files=sorted(p for pattern in patterns for p in glob.glob(pattern));assert len(files)==28
expected=source['nport_identity']['source_xml_files']
sha_map=json.loads((root/'research/stage1_soxx_nport_isin_identity_repair_20261023/result.json').read_text())['original_xml_sha256']
sets={};periods={}
for path in files:
 name=pathlib.Path(path).name
 assert hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()==sha_map[name]
 accession='0001752724-19-177706' if name=='primary_doc.xml' else re.search(r'\d{10}-\d{2}-\d{6}',name).group()
 tree=ET.parse(path).getroot()
 period=next(x.text for x in tree.iter() if x.tag.rsplit('}',1)[-1]=='repPdDate')
 ids=[]
 for node in tree.iter():
  if node.tag.rsplit('}',1)[-1]!='invstOrSec':continue
  fields={x.tag.rsplit('}',1)[-1]:x for x in node}
  value=lambda k:(fields[k].text or '').strip() if k in fields else ''
  if value('assetCat')!='EC' or value('units')!='NS':continue
  ident=fields.get('identifiers')
  vals=[] if ident is None else [x.attrib.get('value','') for x in ident if x.tag.rsplit('}',1)[-1]=='isin']
  assert len(vals)==1 and vals[0]
  ids.append(vals[0])
 assert len(ids)==len(set(ids))==30
 sets[accession]=set(ids);periods[accession]=period
assert set(sets)==set(events.accession) and all(periods[a]==p for a,p in zip(events.accession,events.report_period))
events=events.copy();events['effective_signal_date']=pd.to_datetime(events.effective_signal_date)
assert events.effective_signal_date.isin(cal).all()
rows=[]
for period,g in events.groupby('report_period',sort=True):
 g=g.sort_values('filed')
 if len(g)>1:assert len(g)==2 and sets[g.iloc[0].accession]==sets[g.iloc[1].accession]
 x=g.iloc[0]
 rows.append({'report_period':period,'accession':x.accession,'filed':str(x.filed),'effective_signal_date':str(x.effective_signal_date.date()),'isin_set':sets[x.accession]})
rows.sort(key=lambda x:x['report_period'])
for i,x in enumerate(rows):
 x['entry_count']=None if i==0 else len(x['isin_set']-rows[i-1]['isin_set'])
 x['exit_count']=None if i==0 else len(rows[i-1]['isin_set']-x['isin_set'])
 x['membership_change']=False if i==0 else bool(x['entry_count'] or x['exit_count'])
change_dates=[pd.Timestamp(x['effective_signal_date']) for x in rows if x['membership_change']]
window_counts=[]
for i in range(len(windows)):
 dates=pd.DatetimeIndex(daily.signal_date.iloc[i:i+126])
 window_counts.append(sum(d in dates for d in change_dates))
win=windows.copy();win['change_events_inside']=window_counts;win['nonwinning']=win.relative_log_excess<=0
win['cash_negative']=win.cash_opportunity_log<0
summary={}
for group,frame in [('all',win),('nonwinning',win[win.nonwinning]),('positive',win[~win.nonwinning])]:
 summary[group]={'windows':len(frame),'with_membership_change':int((frame.change_events_inside>0).sum()),'without_membership_change':int((frame.change_events_inside==0).sum()),'cash_negative_without_change':int(((frame.change_events_inside==0)&frame.cash_negative).sum()),'mean_cash_log_without_change':float(frame.loc[frame.change_events_inside==0,'cash_opportunity_log'].mean())}
result={'decision':'descriptive_membership_change_event_support_only_no_policy','plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'original_NPORT_reports':len(events),'distinct_report_periods':len(rows),'amendment_duplicates':len(events)-len(rows),'transitions':len(rows)-1,'transitions_with_membership_change':sum(x['membership_change'] for x in rows),'held_change_event_dates':[x['effective_signal_date'] for x in rows if x['membership_change'] and pd.Timestamp(x['effective_signal_date'])>=pd.Timestamp(daily.signal_date.min()) and pd.Timestamp(x['effective_signal_date'])<=pd.Timestamp(daily.signal_date.max())],'window_groups':summary,'source_event_counts':[{'report_period':x['report_period'],'effective_signal_date':x['effective_signal_date'],'entry_count':x['entry_count'],'exit_count':x['exit_count']} for x in rows],'rolling_windows_overlap':True,'change_event_groups_causal':False,'basket_feature_qualified':False,'market_requests':0,'model_fits':0,'native_accounts':0,'new_Recorders':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k!='source_event_counts'},ensure_ascii=False,indent=2))


[55221:MainThread](2026-10-23 10:17:31,123) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[55221:MainThread](2026-10-23 10:17:31,124) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[55221:MainThread](2026-10-23 10:17:31,124) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{
  "decision": "descriptive_membership_change_event_support_only_no_policy",
  "plan_sha256": "beae1c34bef15f11464a7b5178a73d8a3fa30825fdb5c154302b27a6ee231ae3",
  "original_NPORT_reports": 28,
  "distinct_report_periods": 27,
  "amendment_duplicates": 1,
  "transitions": 26,
  "transitions_with_membership_change": 11,
  "held_change_event_dates": [
    "2023-11-24",
    "2024-11-27",
    "2025-02-28",
    "2025-11-28"
  ],
  "window_groups": {
    "all": {
      "windows": 819,
      "with_membership_change": 439,
      "without_membership_change": 380,
      "cash_negative_without_change": 301,
      "mean_cash_log_without_change": -0.0918389426962665
    },
    "nonwinning": {
      "windows": 564,
      "with_membership_change": 323,
      "without_membership_change": 241,
      "cash_negative_without_change": 241,
      "mean_cash_log_without_change": -0.13080485535185232
    },
    "positive": {
      "windows": 255,
      "with_membership_change": 116,
      "without_membership

In [2]:
# Pure arithmetic on the frozen readback; no new Qlib or source reads.
from pathlib import Path
import json
p=Path('research/stage1_soxx_basket_change_window_support_20261023/result.json')
r=json.loads(p.read_text())
need=574-255
eligible=r['window_groups']['nonwinning']['with_membership_change']
r['event_window_only_static_necessary_repair']={'required_additional_positive_windows':need,'eligible_nonwinning_windows_with_event':eligible,'minimum_fraction_of_eligible_nonwinning_to_flip':need/eligible,'max_positive_if_all_eligible_flip':255+eligible,'not_a_native_account':True,'persistent_effect_outside_event_windows_not_bounded':True}
p.write_text(json.dumps(r,ensure_ascii=False,indent=2)+'\n')
print(r['event_window_only_static_necessary_repair'])


{'required_additional_positive_windows': 319, 'eligible_nonwinning_windows_with_event': 323, 'minimum_fraction_of_eligible_nonwinning_to_flip': 0.9876160990712074, 'max_positive_if_all_eligible_flip': 578, 'not_a_native_account': True, 'persistent_effect_outside_event_windows_not_bounded': True}


## 中文结论

28 份 N-PORT 原件形成 27 个不同报告期、26 个相邻过渡，其中 11 次原始 ISIN 成分集合变化；Stage1 当前 944 会话内只有 4 次在首次披露后 Qlib 信号日可见。819 个连续 126 日窗口中，564 个未赢 SOXX；其中 241 个没有任何成分变化事件，且现金机会成本均为负。若仅改变包含事件的失败窗口，必须使其中至少 319/323 个转正，才达到 574/819 的 70% 门。

这是重叠窗口上的固定路径必要计数，不是因果效应或原生账户。持续持仓状态的跨窗口作用未被此计数排除。成分轮换事件不适合作为唯一修复路线；没有拟合、没有新账户，也未取得 SOXX 超额。